# albucino FP8 PLE overlay for Qwen3.8-Flash-Next W4A16 -> private Kaggle Model (run this ON Kaggle)

**Notebook settings: Accelerator = None (CPU), Internet = ON.** No GPU quota is used. Utility notebook: no
competition data, never submitted.

### Before running: attach two Kaggle Secrets, start the run from the editor
Add-ons -> Secrets -> attach `KAGGLE_USERNAME` and `KAGGLE_KEY` (account **foysalemonshanto**), then
**Save Version -> Save & Run All** from the notebook editor (a run started through the Kaggle API could not reach
the secrets service on 2026-09-26).

### How it works (no disk) — same tested code as the ARC-AGI-3 NVIDIA upload (132.7 GB, 2026-09-26)
Every file streams straight from Hugging Face into Kaggle's resumable upload URL in 256 MiB chunks (RAM only,
3 workers). Each file is hashed while it streams and compared with the pinned Hugging Face tree; the model instance
is created **only if all 21 files match**. A dropped connection resumes from the byte the server confirmed.
The last cell is fail-closed: it raises unless Kaggle lists every file, including the mirror manifest, with the
expected size.

### What gets published (51.2 GB)
Only the files of `albucino/Qwen3.8-Flash-Next-W4A16-FP8PLE` (revision `c6af9fee7a`) that differ from Intel's checkpoint, plus albucino's small
runtime files so that the overlay is authoritative for everything except the weights:
10 FP8 PLE shards (`model-plefp8-*.safetensors`, the official RadixArk FP8 table + scale), `config.json` (Intel's
config + `ple_embedding_dtype: float8_e4m3fn`), `model.safetensors.index.json`, tokenizer / chat template /
generation + preprocessor configs (identical to Intel's), LICENSE, README.md, SHA256SUMS, plus
`KAGGLE_MIRROR_MANIFEST.json`. The 15 main W4A16 shards are byte-identical to
`Intel/Qwen3.8-Flash-Next-W4A16-AutoRound@4c67bf6` (SHA256 checked) and are taken at run time from
`dfranzen/intel-qwen3.8-flash-next-w4a16-autoround` (merged with this overlay by FN-SMOKE).

License: Qwen Community License 1.0 (LICENSE file included). Kaggle label "Other (specified in description)" — the
same label foysalemonshanto's NVIDIA mirror was accepted with (checked 2026-10-05).

Kaggle path afterwards: `/kaggle/input/models/foysalemonshanto/qwen38-flash-next-fp8ple-overlay/pytorch/albucino-fp8ple/1`

The run is idempotent: if the model instance already exists it only lists and verifies its files.


In [ ]:
import hashlib
import json
import os
import queue
import shutil
import subprocess
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import requests

HF_REPO = "albucino/Qwen3.8-Flash-Next-W4A16-FP8PLE"
HF_REVISION = "c6af9fee7ad2cb8ef44b534a99b8c1b81fb0f6da"   # pinned 2026-10-05
HF_URL = "https://huggingface.co/{repo}/resolve/{rev}/{name}"
KAGGLE_MODEL_SLUG = "qwen38-flash-next-fp8ple-overlay"
KAGGLE_MODEL_TITLE = "Qwen3.8 Flash-Next W4A16 FP8-PLE overlay"
KAGGLE_FRAMEWORK = "pyTorch"          # appears as "pytorch" in the /kaggle/input path
KAGGLE_INSTANCE_SLUG = "albucino-fp8ple"
LICENSE_NAME = "Other (specified in description)"   # Qwen Community License 1.0; label accepted for the NVIDIA mirror
PLACEHOLDER_DIR = Path("/tmp/fp8ple-instance-placeholders")   # empty files only: names for model_instance_create
CHUNK = 256 << 20          # bytes per upload request (multiple of 256 KiB, as the resumable protocol requires)
QUANTUM = 256 << 10
READ = 8 << 20             # Hugging Face read size
WORKERS = 3

# name -> (bytes, hash kind, hash). "sha256" = LFS oid for shards, sha256 of the content for the small files
# (the streamer always computes sha256). Taken from the Hugging Face tree API at HF_REVISION.
EXPECTED_FILES = {
    'LICENSE': (3235, 'sha256', 'a0dc422560841fd68e06d974907f8b4c709bca44a67daad2b528437bdf676c08'),
    'README.md': (15547, 'sha256', '3c1ecf16c50cdca78d208dedc5a70cc974e154a83e2e5ea62f6df072cdd37497'),
    'SHA256SUMS': (9453, 'sha256', 'e8a0a6ebd68053b35d28f08e6164354351867c9c9fcf9ada81fee5a557d61b80'),
    'chat_template.jinja': (8952, 'sha256', 'c3cf9e34abf4f9e36c2d72165aa9c132d3e2a725b6c2586aaa3a8af9d7a81041'),
    'config.json': (266471, 'sha256', '2925a0d0989384e0ef7973a14c97ba9c8dfc82c4ca3684d2e83cf4f57db6c172'),
    'generation_config.json': (214, 'sha256', '6ef70d6670f24f03a9041724c7663ef26bc1e678d10f79c93fe11c192ba14b7d'),
    'model-plefp8-00000.safetensors': (5200027199, 'sha256', 'dc2e845b7edd35bda92834fba3626bf7d199e28d6aceac99fee654aade390cfd'),
    'model-plefp8-00001.safetensors': (5200027202, 'sha256', '899eaa0716e28594468a1389ee58cb065c23907f1270de3831f5ecb0a4f82d56'),
    'model-plefp8-00002.safetensors': (5200027198, 'sha256', '06fd8a11abf0419a669f89397b8d70dd6ff42d401e6b2a037c65e49704faaf71'),
    'model-plefp8-00003.safetensors': (5200027189, 'sha256', 'c6aaa1fc08e84eced3c8151ac8679ed943888eba0fcef2556963693430f95bd9'),
    'model-plefp8-00004.safetensors': (5200027190, 'sha256', 'd94e97c96d3ea09208614da016960f3f4b429f47a044c898a51b493f42f74ba2'),
    'model-plefp8-00005.safetensors': (5200027190, 'sha256', '03d5d4792e14a4ab55bae50bb459624b01786f818bbcfcaed1a5d0235af484c6'),
    'model-plefp8-00006.safetensors': (5200027190, 'sha256', '586cccbc12383021bc9bc02f206d9b19fbd1672373ed9a5d91e3b0ce34c2418f'),
    'model-plefp8-00007.safetensors': (5200027190, 'sha256', 'c4a23bcc10f3cde6b633e82c282a9a518e3a64d433628283e1bc592c94cf3d6c'),
    'model-plefp8-00008.safetensors': (5200027190, 'sha256', '2dc8098c0d020bff277c9cf499a6b908e17836b70a5f949dfa24793371c9a87e'),
    'model-plefp8-00009.safetensors': (4400023163, 'sha256', '61de98b89bb79f386795787d7a76827a26f1e292c26edbb0a1b613da455f5a9c'),
    'model.safetensors.index.json': (23657880, 'sha256', 'eb08a649acf89aa2d0c5c1858704cba8e86cdf2b93dec8cf9a89024292a3cc5a'),
    'preprocessor_config.json': (443, 'sha256', '3a159dfec9978a186a72ba085e0ad6a050f3968d8b364218d7bd13f5c89381f2'),
    'tokenizer.json': (19989325, 'sha256', '06b9509352d2af50381ab2247e083b80d32d5c0aba91c272ca9ff729b6a0e523'),
    'tokenizer_config.json': (1165, 'sha256', '792fa3f0cb88b111e54ef3134c873531008c4df471d108da17903426e308aa7b'),
    'video_preprocessor_config.json': (385, 'sha256', '7768af27c1fafa9cc9011c1dc20067e03f8915e03b63504550e11d5066986d13'),
}
EXPECTED_TOTAL_BYTES = 51_244_220_971
NOT_UPLOADED = set()
assert sum(v[0] for v in EXPECTED_FILES.values()) == EXPECTED_TOTAL_BYTES
assert len(EXPECTED_FILES) == 21
print("python", sys.version.split()[0], "| cpus", os.cpu_count(),
      "| /tmp free (df, not the session quota):", round(shutil.disk_usage("/tmp").free / 1e9, 1), "GB")


In [ ]:
# Kaggle Secrets -> Kaggle API credentials. Stops here (before any transfer) if they are missing or wrong.
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    KAGGLE_USERNAME = _secrets.get_secret("KAGGLE_USERNAME").strip()
    _key = _secrets.get_secret("KAGGLE_KEY").strip()
except Exception as exc:
    raise RuntimeError(
        f"Could not read Kaggle Secrets KAGGLE_USERNAME / KAGGLE_KEY ({exc!r}). Attach both (Add-ons -> Secrets) "
        "and start the run from the notebook editor (Save Version -> Save & Run All). A run started through the "
        "Kaggle API got ConnectionError from the secrets service on 2026-09-26.") from None

cred_dir = Path.home() / ".kaggle"
cred_dir.mkdir(parents=True, exist_ok=True)
(cred_dir / "kaggle.json").write_text(json.dumps({"username": KAGGLE_USERNAME, "key": _key}))
(cred_dir / "kaggle.json").chmod(0o600)
os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = _key
del _key

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle==2.2.4"])

from kaggle.api.kaggle_api_extended import KaggleApi
from kaggle.models.upload_file import UploadFile
from kagglesdk.blobs.types.blob_api_service import ApiBlobType, ApiStartBlobUploadRequest

api = KaggleApi()
api.authenticate()
if KAGGLE_USERNAME != "foysalemonshanto":
    raise RuntimeError(f"secrets belong to {KAGGLE_USERNAME!r}, expected foysalemonshanto - nothing transferred")
MODEL_REF = f"{KAGGLE_USERNAME}/{KAGGLE_MODEL_SLUG}"
INSTANCE_REF = f"{MODEL_REF}/pytorch/{KAGGLE_INSTANCE_SLUG}"

# Credential check first: kernels_list(mine=True) answers 401 for a wrong key. (Get-calls cannot be used for
# this: Kaggle answers 403 PERMISSION_DENIED both for a missing model and for someone else's private one.)
try:
    api.kernels_list(mine=True, page_size=1)
except Exception as exc:
    status = getattr(getattr(exc, "response", None), "status_code", None)
    raise RuntimeError(f"Kaggle rejected the KAGGLE_USERNAME / KAGGLE_KEY secrets (HTTP {status}). "
                       "Check the two secret values, then run again.") from None

model_exists = MODEL_REF in {getattr(m, "ref", None) for m in (api.model_list(owner=KAGGLE_USERNAME, page_size=100) or [])}
instance_exists = False
if model_exists:
    try:
        api.model_instance_get(INSTANCE_REF)
        instance_exists = True
    except Exception as exc:
        if getattr(getattr(exc, "response", None), "status_code", None) not in (403, 404):
            raise
print(f"Kaggle API OK as {KAGGLE_USERNAME}; model exists={model_exists}; instance exists={instance_exists}")
if instance_exists:
    print("The instance already exists: the transfer is skipped; the last cell lists its files.")


In [ ]:
# Stream every file Hugging Face -> Kaggle resumable upload, hashing on the fly. Nothing is written to disk.
class _HFReader:
    """Reads one file from a byte offset in a background thread into a bounded queue (<= 48 x 8 MiB)."""

    def __init__(self, url, start, size):
        self.q = queue.Queue(maxsize=48)
        self.stop = threading.Event()
        self.error = None
        self.thread = threading.Thread(target=self._run, args=(url, start, size), daemon=True)
        self.thread.start()

    def _run(self, url, start, size):
        try:
            headers = {"Accept-Encoding": "identity"}
            if start:
                headers["Range"] = f"bytes={start}-"
            with requests.get(url, headers=headers, stream=True, timeout=(30, 300)) as r:
                r.raise_for_status()
                if start and r.status_code != 206:
                    raise RuntimeError(f"Hugging Face ignored the Range header (HTTP {r.status_code})")
                got = start
                for piece in r.iter_content(READ):
                    if self.stop.is_set():
                        return
                    got += len(piece)
                    self.q.put(piece)
                if got != size:
                    raise RuntimeError(f"Hugging Face stream ended at {got} of {size} bytes")
        except BaseException as exc:
            self.error = exc
        finally:
            self.q.put(None)

    def get(self):
        piece = self.q.get()
        if piece is None:
            raise self.error or RuntimeError("Hugging Face stream ended early")
        return piece

    def close(self):
        self.stop.set()
        try:
            while True:
                self.q.get_nowait()
        except queue.Empty:
            pass


def _start_blob(name, size):
    req = ApiStartBlobUploadRequest()
    req.type = ApiBlobType.MODEL
    req.name = name
    req.content_length = size
    req.last_modified_epoch_seconds = int(time.time())
    with api.build_kaggle_client() as kaggle:
        resp = api.with_retry(kaggle.blobs.blob_api_client.start_blob_upload)(req)
    return resp.token, resp.create_url


def _persisted(response, size):
    """Bytes the upload server has stored, from a chunk or status response."""
    if response.status_code in (200, 201):
        return size
    if response.status_code == 308:
        rng = response.headers.get("Range")
        return int(rng.rsplit("-", 1)[1]) + 1 if rng else 0
    raise RuntimeError(f"upload server answered HTTP {response.status_code}: {response.text[:300]}")


def _query(session, put_url, size):
    r = session.put(put_url, headers={"Content-Range": f"bytes */{size}", "Content-Length": "0"}, timeout=(30, 120))
    return _persisted(r, size)


def stream_file(name, size, kind, max_stalls=12):
    """Returns (token, sha256 hex, git-sha1 hex or None, seconds, failures). Gives up only after max_stalls
    consecutive failures without progress (a 53.7 GB file is ~200 requests; isolated resets are expected)."""
    token, put_url = _start_blob(name, size)
    session = requests.Session()
    h256 = hashlib.sha256()
    gsha = hashlib.sha1(b"blob %d\0" % size) if kind == "gitsha1" else None
    hashed = persisted = 0          # hashed == bytes confirmed stored and folded into the digests
    failures = stalls = 0
    stall_mark = -1
    next_report = 5e9
    started = time.time()
    while hashed < size or persisted < size:
        reader = _HFReader(HF_URL.format(repo=HF_REPO, rev=HF_REVISION, name=name), hashed, size)
        buf = bytearray()
        buf_start = hashed
        try:
            while buf_start < size:
                while len(buf) < CHUNK and buf_start + len(buf) < size:
                    buf += reader.get()
                if persisted > buf_start:          # stored by an earlier attempt: hash only, do not resend
                    n = min(persisted - buf_start, len(buf))
                else:
                    at_end = buf_start + len(buf) == size
                    send = len(buf) if at_end else (len(buf) // QUANTUM) * QUANTUM
                    r = session.put(put_url, data=bytes(buf[:send]), timeout=(30, 900),
                                    headers={"Content-Range": f"bytes {buf_start}-{buf_start + send - 1}/{size}"})
                    persisted = _persisted(r, size)
                    n = persisted - buf_start
                    if n < 0 or n > len(buf):
                        raise RuntimeError(f"server reports {persisted} stored; chunk started at {buf_start}")
                with memoryview(buf) as mv:
                    h256.update(mv[:n])
                    if gsha is not None:
                        gsha.update(mv[:n])
                del buf[:n]
                buf_start += n
                hashed = buf_start
                if hashed >= next_report:
                    dt = time.time() - started
                    print(f"    {name}: {hashed / 1e9:.1f}/{size / 1e9:.1f} GB ({hashed / 1e6 / dt:.0f} MB/s)", flush=True)
                    next_report += 5e9
        except Exception as exc:
            failures += 1
            stalls = stalls + 1 if hashed == stall_mark else 1
            stall_mark = hashed
            print(f"    {name}: attempt failed at {hashed} bytes (stall {stalls}/{max_stalls}): {exc!r}"[:400], flush=True)
            if stalls >= max_stalls:
                raise
            session.close()
            session = requests.Session()          # drop pooled connections after a failure
            time.sleep(min(120, 10 * stalls))
            for q in range(5):
                try:
                    persisted = _query(session, put_url, size)
                    break
                except Exception as qexc:
                    if q == 4:
                        raise
                    print(f"    {name}: status query failed ({qexc!r}); retrying"[:300], flush=True)
                    time.sleep(15 * (q + 1))
            if persisted < hashed:
                raise RuntimeError(f"{name}: server now reports {persisted} bytes < {hashed} already confirmed")
        finally:
            reader.close()
    return token, h256.hexdigest(), gsha.hexdigest() if gsha is not None else None, time.time() - started, failures


def upload_bytes(name, data):
    token, put_url = _start_blob(name, len(data))
    r = requests.put(put_url, data=data, headers={"Content-Range": f"bytes 0-{len(data) - 1}/{len(data)}"},
                     timeout=(30, 300))
    if _persisted(r, len(data)) != len(data):
        raise RuntimeError(f"{name}: small upload incomplete")
    return token


TOKENS, HASHES = {}, {}
if not instance_exists:
    todo = sorted((n for n in EXPECTED_FILES if n not in NOT_UPLOADED), key=lambda n: -EXPECTED_FILES[n][0])
    t0 = time.time()

    def _one(name):
        size, kind, want = EXPECTED_FILES[name]
        print(f"[{time.strftime('%H:%M:%S')}] start {name} ({size / 1e9:.2f} GB)", flush=True)
        for attempt in range(1, 4):      # a fresh upload session per attempt (e.g. if the old one expired)
            try:
                token, s256, g1, dt, fails = stream_file(name, size, kind)
                break
            except Exception as exc:
                print(f"[{time.strftime('%H:%M:%S')}] {name}: file attempt {attempt}/3 failed: {exc!r}"[:400], flush=True)
                if attempt == 3:
                    raise
        got = s256 if kind == "sha256" else g1
        print(f"[{time.strftime('%H:%M:%S')}] done  {name} in {dt:.0f} s ({size / 1e6 / max(dt, 1e-6):.0f} MB/s, "
              f"{fails} retries) {kind} {'OK' if got == want else 'MISMATCH'}", flush=True)
        return name, token, s256, got == want

    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        for name, token, s256, ok in pool.map(_one, todo):
            TOKENS[name], HASHES[name] = token, (s256, ok)
    bad = sorted(n for n, (_, ok) in HASHES.items() if not ok)
    total = sum(EXPECTED_FILES[n][0] for n in todo)
    dt = time.time() - t0
    print(f"\nstreamed {len(todo)} files, {total / 1e9:.2f} GB in {dt / 60:.1f} min ({total / 1e6 / dt:.0f} MB/s)")
    if bad:
        raise RuntimeError(f"hash mismatch, instance NOT created: {bad}")
    print(f"all {len(todo)} files match the pinned Hugging Face revision byte-for-byte")

    mirror_manifest = {
        "source": {"hf_repo": HF_REPO, "hf_revision": HF_REVISION},
        "verified_against": "Hugging Face tree API at the pinned revision (LFS sha256 / git blob sha1), "
                            "hashed while streaming",
        "files": {n: {"bytes": EXPECTED_FILES[n][0], "sha256": HASHES[n][0]} for n in sorted(todo)},
        "total_bytes": total,
        "not_uploaded": sorted(NOT_UPLOADED),
        "serving_notes": {
            "what": "overlay: merge with the 15 main shards of Intel/Qwen3.8-Flash-Next-W4A16-AutoRound@4c67bf6 "
                    "(byte-identical to albucino's), e.g. dfranzen/intel-qwen3.8-flash-next-w4a16-autoround on Kaggle",
            "use_from_overlay": "everything except the 15 model-000NN-of-00017 shards",
            "ple": "FP8 E4M3FN n-gram/PLE table + weight_scale (RadixArk), text_config.ple_embedding_dtype=float8_e4m3fn",
            "runtime": "Pennyroyal SGLang 0.5.19 (dfranzen/pennyroyal-v253), --quantization auto-round --ple-offload-embedding",
            "license": "Qwen Community License 1.0 (see LICENSE)",
        },
    }
    MANIFEST_BYTES = (json.dumps(mirror_manifest, indent=1) + "\n").encode("utf-8")
    TOKENS["KAGGLE_MIRROR_MANIFEST.json"] = upload_bytes("KAGGLE_MIRROR_MANIFEST.json", MANIFEST_BYTES)


In [ ]:
# Publish: parent model (if new), then the instance, handing the already-uploaded blob tokens to the library.
if not instance_exists:
    if not model_exists:
        parent_dir = Path("/tmp/kaggle-model-parent")
        shutil.rmtree(parent_dir, ignore_errors=True)
        parent_dir.mkdir(parents=True)
        (parent_dir / "model-metadata.json").write_text(json.dumps({
            "ownerSlug": KAGGLE_USERNAME,
            "title": KAGGLE_MODEL_TITLE,
            "slug": KAGGLE_MODEL_SLUG,
            "subtitle": "FP8 PLE overlay (albucino) for Intel W4A16 AutoRound Qwen3.8-Flash-Next",
            "isPrivate": True,
            "description": (
                f"Private mirror of the FP8-PLE files of {HF_REPO} (revision {HF_REVISION}) for offline Kaggle "
                "inference: 10 FP8 PLE shards + config + index + tokenizer files. Merge with the main shards of "
                "Intel/Qwen3.8-Flash-Next-W4A16-AutoRound. License: Qwen Community License 1.0 (LICENSE in the files)."
            ),
            "licenseName": LICENSE_NAME,
        }, indent=2))
        created = api.model_create_new(str(parent_dir))
        if getattr(created, "error", None):
            raise RuntimeError(f"model create failed: {created.error}")
        print("model created:", getattr(created, "ref", None), getattr(created, "url", None))

    # model_instance_create walks a folder and asks _upload_file for a token per file. The folder holds empty
    # placeholders with the real names; the patched _upload_file returns the token of the streamed blob.
    shutil.rmtree(PLACEHOLDER_DIR, ignore_errors=True)
    PLACEHOLDER_DIR.mkdir(parents=True)
    for name in TOKENS:
        (PLACEHOLDER_DIR / name).touch()
    (PLACEHOLDER_DIR / "model-instance-metadata.json").write_text(json.dumps({
        "ownerSlug": KAGGLE_USERNAME,
        "modelSlug": KAGGLE_MODEL_SLUG,
        "instanceSlug": KAGGLE_INSTANCE_SLUG,
        "framework": KAGGLE_FRAMEWORK,
        "overview": (f"FP8-PLE overlay of {HF_REPO} at {HF_REVISION}, uploaded byte-identical (hashes in "
                     "KAGGLE_MIRROR_MANIFEST.json). 10 model-plefp8 shards + config.json + index + tokenizer files."),
        "usage": ("Symlink these files together with the 15 main shards of Intel W4A16 AutoRound into one folder "
                  "and serve it with Pennyroyal SGLang (--quantization auto-round --ple-offload-embedding). "
                  "License: Qwen Community License 1.0 (LICENSE)."),
        "licenseName": LICENSE_NAME,
        "fineTunable": False,
        "trainingData": [],
        "modelInstanceType": "Unspecified",
        "baseModelInstance": "",
        # "" exactly as in the proven Qwen/Occamy notebooks: the URL field belongs to the EXTERNAL_VARIANT type.
        "externalBaseModelUrl": "",
    }, indent=2))
    used = []
    _original_upload_file = KaggleApi._upload_file

    def _token_for(self, file_name, full_path, blob_type, upload_context, quiet, resources, content_type=None):
        if file_name not in TOKENS:
            raise RuntimeError(f"no streamed blob for {file_name}")
        used.append(file_name)
        return UploadFile(token=TOKENS[file_name])

    KaggleApi._upload_file = _token_for
    try:
        response = api.model_instance_create(str(PLACEHOLDER_DIR), quiet=True, dir_mode="skip")
    finally:
        KaggleApi._upload_file = _original_upload_file
    if sorted(used) != sorted(TOKENS):
        raise RuntimeError(f"token hand-off mismatch: used={sorted(used)}")
    if getattr(response, "error", None):
        raise RuntimeError(f"instance create failed: {response.error}")
    print(f"instance created with {len(used)} files:", getattr(response, "ref", None), getattr(response, "url", None))


In [ ]:
# List what Kaggle now holds (processing can lag a few minutes behind the create call).
expected_upload = {n: EXPECTED_FILES[n][0] for n in EXPECTED_FILES if n not in NOT_UPLOADED}
# the manifest is checked too: exact size when this run uploaded it, presence when the instance already existed
expected_upload["KAGGLE_MIRROR_MANIFEST.json"] = len(MANIFEST_BYTES) if "MANIFEST_BYTES" in globals() else None
listed = {}
for attempt in range(20):
    try:
        resp = api.model_instance_version_files(f"{INSTANCE_REF}/1", page_size=100)
        listed = {f.name: f.size for f in (getattr(resp, "files", None) or [])}
    except Exception as exc:
        print("listing not ready:", repr(exc)[:200])
    if set(expected_upload) <= set(listed):
        break
    time.sleep(60)
print(f"{len(listed)} files listed for {INSTANCE_REF}/1")
for name in sorted(listed):
    print(f"  {listed[name]:>14,}  {name}")
missing = sorted(set(expected_upload) - set(listed))
wrong = {n: (listed[n], expected_upload[n]) for n in expected_upload
         if n in listed and expected_upload[n] is not None and listed[n] != expected_upload[n]}
print("missing:", missing or "none", "| size mismatches:", wrong or "none")
if missing or wrong:
    raise RuntimeError(f"Kaggle mirror verification FAILED: missing={missing} wrong_sizes={wrong} - do not attach this model")
print(f"VERIFIED: all {len(expected_upload)} files listed with the expected sizes")
print("\nattach with:", f'"model_sources": ["{INSTANCE_REF}/1"]')
print("mount path :", f"/kaggle/input/models/{INSTANCE_REF}/1")
